In [1]:
import urllib.request
import os

jdbc_jar = "postgresql-42.6.0.jar"
dest = f"/tmp/spark_jars/{jdbc_jar}"
if not os.path.exists(dest):
    print(f"⬇️  Downloading {jdbc_jar}...")
    urllib.request.urlretrieve(
        "https://repo1.maven.org/maven2/org/postgresql/postgresql/42.6.0/postgresql-42.6.0.jar",
        dest
    )
    print("✅ JDBC jar downloaded")
else:
    print("⏭️  Already exists")

⏭️  Already exists


In [2]:
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--jars /tmp/spark_jars/hadoop-aws-3.3.4.jar,"
    "/tmp/spark_jars/aws-java-sdk-bundle-1.12.262.jar,"
    "/tmp/spark_jars/postgresql-42.6.0.jar "
    "pyspark-shell"
)

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Save_to_Postgres") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

GOLD_PATH = "s3a://rental-observatory/gold/"

# Legge direttamente dal Gold — niente ricalcoli
zip_stress     = spark.read.parquet(GOLD_PATH + "zip_airbnb_stress_summary/")
borough_summary = spark.read.parquet(GOLD_PATH + "airbnb_borough_summary/")
pressure        = spark.read.parquet(GOLD_PATH + "airbnb_pressure/")
market_stress   = spark.read.parquet(GOLD_PATH + "market_rental_stress/")

print("✅ Gold loaded")
zip_stress.show(5)

✅ Gold loaded
+--------+----------------------------+---------------+--------------------+-------------+-----------+----------------+-------------------+-----------------+-----------------+---------------+
|zip_code|neighbourhood_group_cleansed|rent_burden_pct|     stress_category|median_income|market_rent|total_population|num_airbnb_listings|avg_occupancy_pct|avg_host_listings|entire_home_pct|
+--------+----------------------------+---------------+--------------------+-------------+-----------+----------------+-------------------+-----------------+-----------------+---------------+
|   10454|                       Bronx|         147.17|🔴 Severely Stressed|        24086|  2954.0105|           39570|                 56|             40.4|              1.8|           35.7|
|   10002|                   Manhattan|         107.77|🔴 Severely Stressed|        48386|  4345.4795|           76873|                810|             41.0|             45.7|           62.0|
|   10456|                  

In [4]:
# Configurazione connessione PostgreSQL
PG_URL = "jdbc:postgresql://postgres:5432/rental_observatory"
PG_PROPS = {
    "user": "bdt_admin",
    "password": "bdt_password",
    "driver": "org.postgresql.Driver"
}

print("💾 Saving to PostgreSQL...")

# Tabella principale — zip stress + airbnb
zip_stress.write \
    .jdbc(PG_URL, "zip_airbnb_stress", mode="overwrite", properties=PG_PROPS)
print("✅ zip_airbnb_stress saved")

# Borough summary
borough_summary.write \
    .jdbc(PG_URL, "airbnb_borough_summary", mode="overwrite", properties=PG_PROPS)
print("✅ airbnb_borough_summary saved")

# Pressure index
pressure.write \
    .jdbc(PG_URL, "airbnb_pressure", mode="overwrite", properties=PG_PROPS)
print("✅ airbnb_pressure saved")

# Market stress per ZIP
market_stress.write \
    .jdbc(PG_URL, "market_rental_stress", mode="overwrite", properties=PG_PROPS)
print("✅ market_rental_stress saved")

print("\n🎉 All tables saved to PostgreSQL!")

💾 Saving to PostgreSQL...
✅ zip_airbnb_stress saved
✅ airbnb_borough_summary saved
✅ airbnb_pressure saved
✅ market_rental_stress saved

🎉 All tables saved to PostgreSQL!
